# EXP-006: Multi-seed (마무리 병기)

**근거:** EXP-005 Driver FE 실패. 다양성 회복 길로 multi-seed 도입.
- Multi-seed는 같은 모델의 stochastic 분산을 평균으로 줄임 → OOF 안정 + 미세 LB 개선 보장에 가까움
- KFold split seed=42 고정 (OOF concat 가능)
- 각 모델 random_state ∈ {42, 123, 2026} → fold당 3개 모델, OOF/test_pred 평균
- **나머지 설정 EXP-004와 동일** (lr=0.02, GPU XGB/CAT, CPU LGB)

**예상 시간**: 약 2시간 30분 (CAT 3 seed × 45분 ≈ 2:15 + others)

In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

TARGET = 'PitNextLap'
CAT_COLS = ['Driver', 'Compound', 'Race']
y = train[TARGET].astype(int).values

train_le = train.copy(); test_le = test.copy()
for c in CAT_COLS:
    le = LabelEncoder()
    train_le[c] = le.fit_transform(train_le[c].astype(str))
    seen = set(le.classes_)
    test_le[c] = test_le[c].astype(str).map(lambda v: le.transform([v])[0] if v in seen else -1)

train_cb = train.copy(); test_cb = test.copy()
for c in CAT_COLS:
    train_cb[c] = train_cb[c].astype(str)
    test_cb[c]  = test_cb[c].astype(str)

drop_cols = ['id', TARGET]
feature_cols = [c for c in train.columns if c not in drop_cols]
X_le, X_test_le = train_le[feature_cols], test_le[feature_cols]
X_cb, X_test_cb = train_cb[feature_cols], test_cb[feature_cols]

N_FOLDS, KFOLD_SEED = 5, 42
SEEDS = [42, 123, 2026]
LR = 0.02
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=KFOLD_SEED)
splits = list(skf.split(X_le, y))
print(f'Train: {train.shape}, Test: {test.shape}, Seeds: {SEEDS}')

Train: (439140, 16), Test: (188165, 15), Seeds: [42, 123, 2026]


## 1. XGB multi-seed (GPU, lr=0.02)

In [2]:
from xgboost import XGBClassifier

oof_xgb = np.zeros(len(X_le))
test_xgb = np.zeros(len(X_test_le))
xgb_iters = []
t0 = time.time()
for s_i, seed in enumerate(SEEDS):
    seed_oof = np.zeros(len(X_le))
    for fold, (tr_idx, va_idx) in enumerate(splits):
        m = XGBClassifier(
            n_estimators=5000, max_depth=8, learning_rate=LR,
            subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
            gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
            objective='binary:logistic', eval_metric='auc',
            device='cuda', tree_method='hist',
            random_state=seed, verbosity=0,
            early_stopping_rounds=200,
        )
        m.fit(X_le.iloc[tr_idx], y[tr_idx],
              eval_set=[(X_le.iloc[va_idx], y[va_idx])], verbose=0)
        seed_oof[va_idx] = m.predict_proba(X_le.iloc[va_idx])[:, 1]
        test_xgb += m.predict_proba(X_test_le)[:, 1] / N_FOLDS / len(SEEDS)
        xgb_iters.append(m.best_iteration)
    auc_seed = roc_auc_score(y, seed_oof)
    print(f'XGB seed={seed}: OOF AUC={auc_seed:.5f}')
    oof_xgb += seed_oof / len(SEEDS)

xgb_oof_auc = roc_auc_score(y, oof_xgb)
print(f'XGB multi-seed OOF AUC: {xgb_oof_auc:.5f}  (best_iter mean {np.mean(xgb_iters):.0f}, elapsed {time.time()-t0:.1f}s)')

XGB seed=42: OOF AUC=0.94946
XGB seed=123: OOF AUC=0.94952
XGB seed=2026: OOF AUC=0.94952
XGB multi-seed OOF AUC: 0.94969  (best_iter mean 2382, elapsed 219.0s)


## 2. LGB multi-seed (CPU, lr=0.02)

In [3]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

oof_lgb = np.zeros(len(X_le))
test_lgb = np.zeros(len(X_test_le))
lgb_iters = []
t0 = time.time()
for s_i, seed in enumerate(SEEDS):
    seed_oof = np.zeros(len(X_le))
    for fold, (tr_idx, va_idx) in enumerate(splits):
        m = LGBMClassifier(
            n_estimators=5000, learning_rate=LR, num_leaves=64, max_depth=-1,
            min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=1.0,
            objective='binary', metric='auc',
            random_state=seed, n_jobs=-1, verbose=-1,
        )
        m.fit(X_le.iloc[tr_idx], y[tr_idx],
              eval_set=[(X_le.iloc[va_idx], y[va_idx])],
              callbacks=[early_stopping(200, verbose=False), log_evaluation(0)])
        seed_oof[va_idx] = m.predict_proba(X_le.iloc[va_idx])[:, 1]
        test_lgb += m.predict_proba(X_test_le)[:, 1] / N_FOLDS / len(SEEDS)
        lgb_iters.append(m.best_iteration_)
    auc_seed = roc_auc_score(y, seed_oof)
    print(f'LGB seed={seed}: OOF AUC={auc_seed:.5f}')
    oof_lgb += seed_oof / len(SEEDS)

lgb_oof_auc = roc_auc_score(y, oof_lgb)
print(f'LGB multi-seed OOF AUC: {lgb_oof_auc:.5f}  (best_iter mean {np.mean(lgb_iters):.0f}, elapsed {time.time()-t0:.1f}s)')

LGB seed=42: OOF AUC=0.94933
LGB seed=123: OOF AUC=0.94935
LGB seed=2026: OOF AUC=0.94934
LGB multi-seed OOF AUC: 0.94959  (best_iter mean 3901, elapsed 370.4s)


## 3. CatBoost multi-seed (GPU, lr=0.02)

In [4]:
from catboost import CatBoostClassifier

cat_idx = [feature_cols.index(c) for c in CAT_COLS]

oof_cat = np.zeros(len(X_cb))
test_cat = np.zeros(len(X_test_cb))
cat_iters = []
t0 = time.time()
for s_i, seed in enumerate(SEEDS):
    seed_oof = np.zeros(len(X_cb))
    for fold, (tr_idx, va_idx) in enumerate(splits):
        m = CatBoostClassifier(
            iterations=20000, learning_rate=LR, depth=8, l2_leaf_reg=3.0,
            bagging_temperature=0.5, random_strength=1,
            eval_metric='AUC', loss_function='Logloss',
            cat_features=cat_idx, random_state=seed,
            verbose=False, allow_writing_files=False,
            task_type='GPU', devices='0',
            early_stopping_rounds=200,
        )
        m.fit(X_cb.iloc[tr_idx], y[tr_idx],
              eval_set=(X_cb.iloc[va_idx], y[va_idx]),
              use_best_model=True)
        seed_oof[va_idx] = m.predict_proba(X_cb.iloc[va_idx])[:, 1]
        test_cat += m.predict_proba(X_test_cb)[:, 1] / N_FOLDS / len(SEEDS)
        cat_iters.append(m.get_best_iteration())
    auc_seed = roc_auc_score(y, seed_oof)
    print(f'CAT seed={seed}: OOF AUC={auc_seed:.5f}, elapsed so far={time.time()-t0:.0f}s')
    oof_cat += seed_oof / len(SEEDS)

cat_oof_auc = roc_auc_score(y, oof_cat)
print(f'CAT multi-seed OOF AUC: {cat_oof_auc:.5f}  (best_iter mean {np.mean(cat_iters):.0f}, elapsed {time.time()-t0:.1f}s)')

Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


CAT seed=42: OOF AUC=0.94952, elapsed so far=1953s


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


CAT seed=123: OOF AUC=0.94949, elapsed so far=3821s


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


CAT seed=2026: OOF AUC=0.94959, elapsed so far=6027s
CAT multi-seed OOF AUC: 0.94978  (best_iter mean 8682, elapsed 6027.3s)


## 4. Blending

In [5]:
oof_blend_eq  = (oof_xgb + oof_lgb + oof_cat) / 3
test_blend_eq = (test_xgb + test_lgb + test_cat) / 3
auc_eq = roc_auc_score(y, oof_blend_eq)

best = (None, -1)
for w_xgb in np.arange(0, 1.001, 0.05):
    for w_lgb in np.arange(0, 1.001 - w_xgb, 0.05):
        w_cat = 1 - w_xgb - w_lgb
        if w_cat < 0: continue
        s = w_xgb * oof_xgb + w_lgb * oof_lgb + w_cat * oof_cat
        a = roc_auc_score(y, s)
        if a > best[1]:
            best = ((round(w_xgb, 2), round(w_lgb, 2), round(w_cat, 2)), a)

w_opt, auc_opt = best
oof_blend_opt  = w_opt[0] * oof_xgb + w_opt[1] * oof_lgb + w_opt[2] * oof_cat
test_blend_opt = w_opt[0] * test_xgb + w_opt[1] * test_lgb + w_opt[2] * test_cat

print('=== EXP-006 결과 (multi-seed × 3) ===')
print(f'XGB         OOF AUC: {xgb_oof_auc:.5f}')
print(f'LGB         OOF AUC: {lgb_oof_auc:.5f}')
print(f'CAT         OOF AUC: {cat_oof_auc:.5f}')
print(f'Blend equal       :  {auc_eq:.5f}')
print(f'Blend opt {w_opt}: {auc_opt:.5f}')
print()
print('=== vs EXP-004 (single seed) ===')
print(f'  XGB   0.94946 -> {xgb_oof_auc:.5f}  ({xgb_oof_auc-0.94946:+.5f})')
print(f'  LGB   0.94933 -> {lgb_oof_auc:.5f}  ({lgb_oof_auc-0.94933:+.5f})')
print(f'  CAT   0.94961 -> {cat_oof_auc:.5f}  ({cat_oof_auc-0.94961:+.5f})')
print(f'  Blend 0.95042 -> {auc_eq:.5f}      ({auc_eq-0.95042:+.5f})')
print(f'  Opt   0.95048 -> {auc_opt:.5f}     ({auc_opt-0.95048:+.5f})')

=== EXP-006 결과 (multi-seed × 3) ===
XGB         OOF AUC: 0.94969
LGB         OOF AUC: 0.94959
CAT         OOF AUC: 0.94978
Blend equal       :  0.95048
Blend opt (np.float64(0.25), np.float64(0.25), np.float64(0.5)): 0.95053

=== vs EXP-004 (single seed) ===
  XGB   0.94946 -> 0.94969  (+0.00023)
  LGB   0.94933 -> 0.94959  (+0.00026)
  CAT   0.94961 -> 0.94978  (+0.00017)
  Blend 0.95042 -> 0.95048      (+0.00006)
  Opt   0.95048 -> 0.95053     (+0.00005)


## 5. Submissions

In [6]:
for name, prob in [
    ('exp006_xgb',         test_xgb),
    ('exp006_lgb',         test_lgb),
    ('exp006_cat',         test_cat),
    ('exp006_blend_equal', test_blend_eq),
    ('exp006_blend_opt',   test_blend_opt),
]:
    sub = pd.DataFrame({'id': test['id'], 'PitNextLap': prob})
    path = f'../submissions/submission_{name}.csv'
    sub.to_csv(path, index=False)
    print(f'  saved {path}  mean={prob.mean():.4f}')

  saved ../submissions/submission_exp006_xgb.csv  mean=0.1973
  saved ../submissions/submission_exp006_lgb.csv  mean=0.1972
  saved ../submissions/submission_exp006_cat.csv  mean=0.1978
  saved ../submissions/submission_exp006_blend_equal.csv  mean=0.1974
  saved ../submissions/submission_exp006_blend_opt.csv  mean=0.1975
